In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver



In [ ]:
load_dotenv()

In [ ]:
model = ChatGoogleGenerativeAI(model='gemini-flash-2.5')

In [ ]:
def chat_node(state: MessagesState):
    response = model.invoke(state['messages'])
    return {'messages': [response]}

In [ ]:
checkpointer = InMemorySaver()

In [ ]:
graph = StateGraph(MessagesState)
graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

workflow = graph.compile(checkpointer=checkpointer)
workflow

In [ ]:
CONFIG = {"configurable": {"thread_id": "thread-1"}}

In [ ]:
result = workflow.invoke({"messages": [{"role": "user", "content": "Hi! My name is Harris."}]}, config)

In [ ]:
snap = graph.get_state(config)
vals = snap.values
for m in vals.get("messages", []):
        print("-", type(m).__name__, ":", m.content)